# 12. pandas DataFrames — `pandas`
**Python Libraries Lab · Data Science**

_Load, clean, filter, group and summarise tables of data._

pandas is Excel inside Python, but faster and repeatable. Its main object is the DataFrame, a table with named columns. Each column is a Series. You can load a CSV in one line, then filter, sort, group and summarise it with short readable commands. Every analysis is saved as code, so you can rerun it next month in seconds.

**In this notebook:** You will analyse six hundred ShopEasy orders. Which city brings the most revenue, which category has the best ratings, how sales change month by month, and you will join a second table of city managers.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
Colab already includes this library. This cell just makes sure it is there (it takes a few seconds).

In [ ]:
!pip install -q pandas
import importlib.metadata as md
print("pandas version:", md.version("pandas"))

## Step 1 · Create the ShopEasy dataset
This makes 600 realistic orders and saves them as `shop_orders.csv`. Several lessons reuse this exact dataset.

In [ ]:
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
n = 600
orders = pd.DataFrame({
    "order_id": np.arange(1001, 1001 + n),
    "date": pd.Timestamp("2026-01-01") + pd.to_timedelta(rng.integers(0, 90, n), unit="D"),
    "city": rng.choice(["Mumbai", "Delhi", "Pune", "Bengaluru"], n),
    "category": rng.choice(["Books", "Electronics", "Clothing", "Home"], n, p=[0.3, 0.2, 0.3, 0.2]),
    "quantity": rng.integers(1, 6, n),
    "price": rng.choice([199, 499, 999, 1499, 2999], n),
    "rating": rng.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.1, 0.2, 0.35, 0.3]),
})
orders["revenue"] = orders["quantity"] * orders["price"]
orders.to_csv("shop_orders.csv", index=False)
print(orders.shape, "-> saved shop_orders.csv")
orders.head()

## Step 2 · Load and look
`read_csv` with `parse_dates` turns the date column into real dates.

In [ ]:
df = pd.read_csv("shop_orders.csv", parse_dates=["date"])
print(df.shape)
df.info()
df.describe().round(1)

## Step 3 · Select and filter
Pick columns with a list. Filter rows with a condition; combine conditions with `&` and `|` in brackets.

In [ ]:
print(df[["city", "category", "revenue"]].head(3))

big_pune = df[(df["city"] == "Pune") & (df["revenue"] > 5000)]
print("\nBig Pune orders:", len(big_pune))
big_pune.sort_values("revenue", ascending=False).head()

## Step 4 · groupby: answer business questions
Split by a column, apply a summary, combine the results.

In [ ]:
by_city = (df.groupby("city")
            .agg(orders=("order_id", "count"), revenue=("revenue", "sum"), avg_rating=("rating", "mean"))
            .sort_values("revenue", ascending=False)
            .round(2))
by_city

## Step 5 · New columns and dates
Create columns from others. The `.dt` accessor pulls month, weekday and more.

In [ ]:
df["month"] = df["date"].dt.month_name()
df["is_weekend"] = df["date"].dt.dayofweek >= 5
df["band"] = pd.cut(df["revenue"], bins=[0, 1000, 5000, 20000], labels=["small", "medium", "large"])

print(df.groupby("month", sort=False)["revenue"].sum())
print()
print(df["band"].value_counts())

## Step 6 · Pivot tables
Like Excel pivots: rows, columns and a value to summarise.

In [ ]:
pd.pivot_table(df, index="city", columns="category", values="revenue", aggfunc="sum", margins=True)

## Step 7 · Missing values and merging
`isna`, `fillna` and `dropna` handle gaps. `merge` joins two tables on a shared column.

In [ ]:
managers = pd.DataFrame({"city": ["Mumbai", "Delhi", "Pune"], "manager": ["Riya", "Aman", "Neha"]})
merged = by_city.reset_index().merge(managers, on="city", how="left")
print(merged)
print("\nMissing per column:\n", merged.isna().sum())
merged["manager"] = merged["manager"].fillna("Hiring")
merged

## Step 8 · Save your results
Write to CSV or Excel. In Colab, use the Files panel (left sidebar) to download.

In [ ]:
merged.to_csv("city_report.csv", index=False)
print(open("city_report.csv").read())

## ✍️ Your turn
Using `df`, store the **category with the highest average rating** (just its name) in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    expected = df.groupby("category")["rating"].mean().idxmax()
    return answer == expected, f"Expected {expected!r}, got {answer!r}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = df.groupby("category")["rating"].mean().idxmax()

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **DataFrame**
- **Series**
- **loc / iloc**
- **groupby**
- **merge**
- **pivot_table**

**Next:** 13. Scientific Toolkit (`scipy`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/13_scipy.ipynb)

## 💻 Run it on your own computer (optional)
1. Install Python 3.10+ from python.org (on Windows tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install pandas
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that just opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain `.py` file, run `!` commands in the terminal instead.